In [52]:
import brainsss
import os
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import colors
%matplotlib inline
#from sklearn.cluster import AgglomerativeClustering
import scipy
import time
import h5py
import ants
import nibabel as nib
from scipy.ndimage import uniform_filter, gaussian_filter
import shutil
from sklearn.cluster import AgglomerativeClustering
from sklearn.feature_extraction.image import grid_to_graph
import gc
import sys
import warnings
from scipy.ndimage import gaussian_filter1d,gaussian_filter
from scipy.signal import butter, sosfiltfilt, filtfilt, freqz,iirnotch
import cv2
from scipy.ndimage.morphology import binary_erosion
from scipy.ndimage.morphology import binary_dilation
from scipy.ndimage import zoom
from sklearn.mixture import GaussianMixture
import scipy.stats as stats
import sklearn
import pickle
import itertools
from statsmodels.stats.multitest import multipletests
import seaborn as sns
import pandas as pd
from skimage import measure
from mpl_toolkits.mplot3d import Axes3D
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from scipy.ndimage.morphology import binary_erosion, binary_dilation
import scipy.cluster.hierarchy as sch
from scipy.spatial.distance import pdist
import csv
from functools import reduce
from sklearn.decomposition import PCA
import multiprocessing as mp
from multiprocessing import Pool
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.multiclass import OneVsRestClassifier
import scour.scour
import gzip

In [53]:
color_map_color_1=np.asarray(['#780000','#C1121F','#E0898F','#F0C4C7','#FFFFFF','#D9E6EF','#B3CDDE','#669BBC', '#003049'])[::-1]
cmap_personal = matplotlib.colors.LinearSegmentedColormap.from_list(
    'cmap', color_map_color_1)

In [54]:
home_path = '/oak/stanford/groups/trc/data/Ilana/2P/data/later/'
later_dir = '/oak/stanford/groups/trc/data/Ilana/2P/data/later/temp_filter'
cluster_dir = os.path.join(later_dir, 'clustering')
save_dir=os.path.join(later_dir,'figs')
n_clusters=500
# Suppress RuntimeWarning
warnings.filterwarnings('ignore', category=RuntimeWarning)

In [55]:
%%time
event='fixed_10flies_final'
total_path = os.path.join(home_path, f'{event}_event_times_split_dic.pkl')
print("Loading total dict")
with open(total_path, 'rb') as file:
    total_data_dict = pickle.load(file)

print(total_data_dict.keys())
print(total_data_dict['250'].keys())

event='fixed_best_final'
total_path = os.path.join(home_path, f'{event}_event_times_split_dic.pkl')
print("Loading total dict")
with open(total_path, 'rb') as file:
    best_data_dict = pickle.load(file)

print(best_data_dict.keys())
print(best_data_dict['250'].keys())

Loading total dict
dict_keys(['226', '227', '228', '234', '239', '240', '241', '242', '249', '250'])
dict_keys(['total', 'inc', 'dec', 'flat', 'non'])
Loading total dict
dict_keys(['226', '227', '228', '234', '239', '240', '241', '242', '249', '250'])
dict_keys(['total', 'inc', 'dec', 'flat', 'non'])
CPU times: user 5.5 ms, sys: 142 µs, total: 5.64 ms
Wall time: 1.71 s


In [56]:
for file in os.listdir(cluster_dir):
#     print(file)
    if f'_{n_clusters}' in file and 'labels' in file and 'total' in file:
        print(file)
        label_files=os.path.join(cluster_dir,file)
giant_total_labels=np.load(label_files)
print(giant_total_labels.shape)

supercluster_labels_total_500.npy
(4171804,)


In [57]:
label_size=np.zeros(n_clusters)
for i,label in enumerate(np.sort(np.unique(giant_total_labels))):
    label_size[i]=np.count_nonzero(np.isin(giant_total_labels,label))
max_size_clus=np.where(label_size==np.max(label_size))[0][0]
print(f'the biggest cluster (and therefore the background) is cluster {max_size_clus}')

the biggest cluster (and therefore the background) is cluster 71


In [58]:
for i,label in enumerate(giant_total_labels):
    if label==max_size_clus:
        giant_total_labels[i]=-100
    elif label>max_size_clus:
        giant_total_labels[i]=label-1

In [59]:
giant_labels_anat=giant_total_labels.reshape(314,146,91)
print(giant_labels_anat.shape)

(314, 146, 91)


In [60]:
#ts is in 10ms units

In [85]:
total_path = os.path.join(home_path, 'behave_all','behavior_all.pkl')
with open(total_path, 'rb') as file:
    total_data_dict_2 = pickle.load(file)

In [86]:
total_event_times_dic={}
total_trials={'inc':[],'dec':[],'flat':[],'non':[]}
total_idx={}
total_behavior={'total':[],'dec':[],'inc':[],'flat':[],'non':[]}
behave_lst={}
for fly in total_data_dict_2:
    total_event_times_dic[fly]={'inc':[],'dec':[],'flat':[],'non':[]}
    total_idx[fly]={'inc':[],'dec':[],'flat':[],'non':[]}
    fly_behavior=total_data_dict_2[fly]['behavior']['Y']
    looms=total_data_dict_2[fly]['event_times']
    behave_lst[fly]={'total':[],'label':[]}
    for i,trial in enumerate(fly_behavior):
    #     print(np.shape(trial))
        pre=np.mean(trial[150:200])
        post=np.mean(trial[250:300])
        total_behavior['total'].append([pre, post])
        behave_lst[fly]['total'].append([pre, post])
        if pre>-0.1 and pre<0.1 and post>-0.1 and post<0.1:
                total_event_times_dic[fly]['flat'].append(looms[i])
                total_trials['flat'].append(fly_behavior[i])
                total_behavior['flat'].append([pre, post])
                total_idx[fly]['flat'].append(i)
                b='flat'
        elif post>np.abs(pre*2):
                total_event_times_dic[fly]['inc'].append(looms[i])
                total_trials['inc'].append(fly_behavior[i])
                total_behavior['inc'].append([pre, post])
                total_idx[fly]['inc'].append(i)
                b='inc'
        elif post<pre/2:
                total_event_times_dic[fly]['dec'].append(looms[i])
                total_trials['dec'].append(fly_behavior[i])
                total_behavior['dec'].append([pre, post])
                total_idx[fly]['dec'].append(i)
                b='dec'
                
        else:
            total_event_times_dic[fly]['non'].append(looms[i])
            total_trials['non'].append(fly_behavior[i])
            total_behavior['non'].append([pre, post])
            total_idx[fly]['non'].append(i)
            b='non'
        behave_lst[fly]['label'].append(b)
    behave_lst[fly]['label']=np.asarray(behave_lst[fly]['label'])
    behave_lst[fly]['total']=np.asarray(behave_lst[fly]['total'])

In [87]:
np.shape(behave_lst[fly]['total'])

(199, 2)

In [88]:
%%time
total_events_behave={}
for fly in fly_clust:
    print(f'fly is {fly}')
    total_events_behave[fly]={'vals':[],'label':[],'fv':[]}
    for cluster in fly_clust[fly]:
#         print(f'cluster is {cluster}')
        if cluster!=max_size_clus:
            tempv=[]
            templ=[]
            tempf=[]
            for i,event in enumerate(behave_lst[str(fly)]['label']):
                try:
#                 print(f'event is {event}, and idx is {i}, and cluster is {cluster}')
                    tempv.append(fly_clust[fly][cluster][i][-1])
                    templ.append(event)
                    tempf.append(behave_lst[str(fly)]['total'][i])
#                 print(np.shape(tempf))
                except:
                    continue
            total_events_behave[fly]['vals'].append(tempv)
            total_events_behave[fly]['label'].append(templ)
    total_events_behave[fly]['vals']=np.vstack(total_events_behave[fly]['vals'])
    total_events_behave[fly]['label']=np.vstack(total_events_behave[fly]['label'])[0]
    total_events_behave[fly]['fv']=np.vstack(tempf)
#     total_events_behave[fly]['event_num']=np.vstack(total_events_behave[fly]['event_num'])

fly is 226
fly is 227
fly is 228
fly is 234
fly is 239
fly is 240
fly is 241
fly is 242
fly is 249
fly is 250
CPU times: user 1.42 s, sys: 11.8 ms, total: 1.43 s
Wall time: 1.43 s


In [61]:
fly_nums=list(best_data_dict.keys())
print(fly_nums)

['226', '227', '228', '234', '239', '240', '241', '242', '249', '250']


In [62]:
def count_trial_type(data_dict):
    flies=list(data_dict.keys())
    be=list(data_dict[flies[0]].keys())
    count_new={}
    for b in be:
        count_new[b]=0
    for fly in data_dict:
        for b in data_dict[fly]:
            c=np.shape(data_dict[fly][b])[0]
            count_new[b]+=c
    for b in be:
        print(f'{b} {count_new[b]}')

In [63]:
count_trial_type(best_data_dict)

total 1982
inc 133
dec 144
flat 526
non 1179


In [64]:
def make_similar_counts(data_dict):
    behaviors=list(data_dict[fly_nums[-1]].keys())
    count_r={}
    for b in behaviors:
        count_r[b]=0
        for fly in data_dict:
            c=np.shape(data_dict[fly][b])[0]
            count_r[b]+=c
    min_count=np.min(list(count_r.values()))
    for fly in data_dict:
        for b in data_dict[fly]:
            num_trials=count_r[b]
            per=min_count/num_trials
            num_current=np.shape(data_dict[fly][b])[0]
            num_take=int(num_current*per)
    #         print(num_current, num_take, per)
            arr=np.random.choice(np.arange(num_current),num_take,replace=False)
            arr = np.sort(np.asarray(arr).astype(int))
            data_dict[fly][b]=np.asarray(data_dict[fly][b])[arr]
    return data_dict

In [65]:
best_data_dict_new=make_similar_counts(best_data_dict)

In [66]:
count_trial_type(best_data_dict_new)

total 130
inc 133
dec 129
flat 128
non 128


In [67]:
count_trial_type(total_data_dict)

total 1982
inc 386
dec 697
flat 526
non 373


In [68]:
new_total_data_dict={}
for fly in total_data_dict:
    print(fly)
    new_total_data_dict[str(fly)]={}
    for b in total_data_dict[fly]:
        if b != 'total':
            bool_sp=~np.isin(total_data_dict[fly][b],best_data_dict[fly][b])
#             print(fly, b, np.shape(bool_sp))
            new_total_data_dict[fly][b]=np.asarray(total_data_dict[fly][b])[bool_sp]

226
227
228
234
239
240
241
242
249
250


In [69]:
new_total_data_dict=make_similar_counts(new_total_data_dict)

In [70]:
count_trial_type(new_total_data_dict)

inc 253
dec 248
flat 247
non 248


In [71]:
# test_fly_nums=['226','227','228']#,'234','239']
fly_clust={}
for fly in fly_nums:
    fly_path=os.path.join(home_path,f'{fly}_individual_clusters_new_ch_2_dict.pkl') # this name changed
    if os.path.exists(fly_path):
        print(f'fly is {fly}')
        with open(fly_path, 'rb') as file:
            fly_ind_dict = pickle.load(file)
            fly_clust[fly]=fly_ind_dict
    else:
        print(f'fly {fly} not there yet!')


fly is 226
fly is 227
fly is 228
fly is 234
fly is 239
fly is 240
fly is 241
fly is 242
fly is 249
fly is 250


In [72]:
def make_train_test_events(total_behave_dict, other_behave_dict):
    event_labels={}
    fly_nums=list(other_behave_dict.keys())
    for fly in fly_nums:
        event_labels[fly]={}
        trials=np.hstack(list(other_behave_dict[fly].values()))
        tn=np.shape(trials)[0]
#         print(fly, tn)
        event_labels[fly]['label']=np.full(tn,str)
        event_labels[fly]['idx']=np.full(tn,int)
        event_labels[fly]['time']=np.full(tn,int)
        time=0
        for behave in other_behave_dict[fly]:
            for i in range(np.shape(other_behave_dict[fly][behave])[0]):
    #             print(f'{fly}: {behave}, {best_data_dict[fly][behave][i]}')
                if np.isin(total_behave_dict[fly]['total'],other_behave_dict[fly][behave][i]).any()==True:
                    place=np.where(np.isin(total_behave_dict[fly]['total'],other_behave_dict[fly][behave][i]))[0][0]
                    event_labels[fly]['label'][time]=behave
                    event_labels[fly]['idx'][time]=place
                    event_labels[fly]['time'][time]=total_behave_dict[fly]['total'][place]
                    time+=1
                else:
                    print(f'{fly}: {behave} {best_data_dict[fly][behave][i]} does not exist in total')
    return event_labels

In [73]:
def seperate_events_by_cluster(n_clusters,max_size_clust,fly_clust, total_behave_dict, other_behave_dict, sep=True):
    event_labels=make_train_test_events(total_behave_dict, other_behave_dict)
    cluster_events={}
    range_r=np.arange(n_clusters)
    for cluster in range(n_clusters):
        if cluster!=max_size_clus:
            cluster_events[cluster]={'vals':[],'label':[]}
    for fly in fly_clust:
        print(f'fly is {fly}')
        for cluster in fly_clust[fly]:
#             print(f'cluster is {cluster}')
            if cluster!=max_size_clus:
                for i,event in enumerate(event_labels[fly]['idx']):
    #                 print(f'event is {event}, and idx is {i}')
                    cluster_events[cluster]['vals'].append(np.nanmean(fly_clust[fly][cluster][event][-1]))
                    cluster_events[cluster]['label'].append(event_labels[str(fly)]['label'][i])
            
    if sep:
        sep_events={}
        for key in cluster_events:
            sep_events[key]={}
            labels=np.asarray(cluster_events[key]['label'])
#             print(np.shape(labels))
            labels_idx=np.where((labels == 'inc') | (labels == 'flat') | (labels == 'dec'))
            
            data_labels=np.asarray(cluster_events[key]['label'])[labels_idx]
            data_vals=np.asarray(cluster_events[key]['vals'])[labels_idx]
            sep_events[key]['vals']=data_vals
            sep_events[key]['label']=data_labels
    return sep_events

In [74]:
best_events=seperate_events_by_cluster(n_clusters,max_size_clus,fly_clust, total_data_dict, best_data_dict_new, sep=True)

fly is 226
fly is 227
fly is 228
fly is 234
fly is 239
fly is 240
fly is 241
fly is 242
fly is 249
fly is 250


In [75]:
total_events=seperate_events_by_cluster(n_clusters,max_size_clus,fly_clust, total_data_dict, new_total_data_dict, sep=True)

fly is 226
fly is 227
fly is 228
fly is 234
fly is 239
fly is 240
fly is 241
fly is 242
fly is 249
fly is 250


In [78]:
def make_X_y(event_dict, max_size_clus, n_clusters):
    X = []
    y = event_dict[0]['label']  # labels should be same across all clusters

    for cluster in range(n_clusters):
        if cluster!=max_size_clus:
            X.append(event_dict[cluster]['vals'])

    X = np.vstack(X).T  # Transpose to get (n_trials, n_clusters)
    print(np.shape(X))
    y = np.array(y)
    print(np.shape(y))
    return X, y

In [79]:
X,y=make_X_y(best_events, max_size_clus, n_clusters)

(390, 499)
(390,)


In [83]:
total_X,total_y=make_X_y(total_events, max_size_clus, n_clusters)

(748, 499)
(748,)


In [ ]:
%%time
model_set={'null_acc':[],
           'train_acc':[],
           'test_acc':[],
           'y_predict':[],
           'y_idx':[],
           'top_100':[],
           'top_import':[],
           'pred':[],
           'behave':[], 
           'total_null':[],
           'total_y':[],
           'total_y_predict':[],
           'total_scores':[],
           'model': []
           
          }
num_permutations=1000
for i in range(num_permutations):
    train_indices, test_indices = train_test_split(
    np.arange(len(y)), 
    test_size=0.2, 
    random_state=i, 
    stratify=y
    )
    rf = RandomForestClassifier(
        n_estimators=200,      # number of trees
        max_depth=6,          # max depth of each tree (prevents overfitting)
        min_samples_split=5,   # minimum samples to split a node
        min_samples_leaf=1,    # minimum samples in leaf
        random_state=i,
        n_jobs=-1              # use all CPU cores
    )
    # Fit model
    rf.fit(X[train_indices], y[train_indices])
    
    # Evaluate
    train_acc = rf.score(X[train_indices], y[train_indices])
    model_set['train_acc'].append(train_acc)
    test_acc = rf.score(X[test_indices], y[test_indices])
    model_set['test_acc'].append(test_acc)
    
    #test null
    num_lab = len(y[test_indices]) 
#     print(num_lab)
    y_null=np.random.choice(list(rf.classes_), size=num_lab)
    null_acc = rf.score(X[test_indices], y_null)
    model_set['null_acc'].append(null_acc)
    
    #pred y
    y_pred = rf.predict(X[test_indices])
    model_set['y_predict'].append(y_pred)
    model_set['y_idx'].append(y[test_indices])
    
    #test total
    test_total_acc = rf.score(total_X, total_y)
    model_set['total_scores'].append(test_total_acc)
    
    #test null total
    num_lab_total = len(total_y) 
#     print(num_lab_total)
    y_total_null=np.random.choice(list(rf.classes_), size=num_lab_total)
    null_acc = rf.score(total_X, y_total_null)
    model_set['total_null'].append(null_acc)
    
    #pred y
    y_pred_total = rf.predict(total_X)
    model_set['total_y_predict'].append(y_pred_total)
    model_set['total_y'].append(total_y)
    
    top_100_idx=np.argsort(rf.feature_importances_)[::-1][:100]
    model_set['top_100'].append(top_100_idx)
    model_set['top_import'].append(rf.feature_importances_[top_100_idx])
    for fly in total_events_behave:
        bool_ar=total_events_behave[fly]['label']!='non'
        pred_X=total_events_behave[fly]['vals'][:,bool_ar]
        fly_predict=rf.predict(pred_X.T)
        model_set['pred'].append(fly_predict)
        model_set['behave'].append(total_events_behave[fly]['fv'][bool_ar])
    

null_acc_final=np.mean(model_set['null_acc'])
train_acc_final=np.mean(model_set['train_acc'])
test_acc_final=np.mean(model_set['test_acc'])
model_set['model']=rf

print(f"Random Forest Performance:")
print(f"Train accuracy: {train_acc_final:.3f}, std: {np.std(train_acc_total)}")
print(f"Test accuracy: {test_acc_final:.3f}, std: {np.std(test_acc_total)}")
print(f"Null accuracy: {null_acc_final:.3f}, std: {np.std(null_scores_total)}")


In [108]:
# file_path=os.path.join(cluster_dir, 'model_set_total_new.pkl')
# with open(file_path, 'wb') as file:
#         pickle.dump(model_set, file)
with open(file_path, 'rb') as file:
    model_set = pickle.load(file)